# 🔍 Clasificador de Paneles Solares - Inferencia

**Notebook ligero para clasificar imágenes usando el modelo ya entrenado.**

- ✅ No requiere re-entrenar el modelo
- ⚡ Carga rápida del modelo guardado
- 📸 Prueba imágenes individuales o múltiples
- 🎯 Predicción: Limpio vs Sucio

## 1. Cargar Librerías

In [13]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import tensorflow as tf
from tensorflow import keras
import ipywidgets as widgets
from IPython.display import display, clear_output
import io

print(f"TensorFlow version: {tf.__version__}")
print("✅ Librerías cargadas")

TensorFlow version: 2.20.0
✅ Librerías cargadas


## 2. Cargar Modelo Entrenado

In [14]:
# Ruta al modelo guardado
MODEL_PATH = Path('artifacts/best_model.keras')

# Verificar que existe
if not MODEL_PATH.exists():
    print("❌ ERROR: No se encontró el modelo.")
    print(f"   Busca el archivo en: {MODEL_PATH}")
    print("   Asegúrate de haber entrenado el modelo primero usando el notebook 'solar_panel_dust_detection.ipynb'")
else:
    # Cargar modelo
    model = keras.models.load_model(MODEL_PATH)
    print(f"✅ Modelo cargado exitosamente desde: {MODEL_PATH}")
    print(f"📊 Parámetros del modelo: {model.count_params():,}")

✅ Modelo cargado exitosamente desde: artifacts/best_model.keras
📊 Parámetros del modelo: 2,427,713


## 3. 🚀 Clasificar Imagen

**Ejecuta la celda siguiente y haz click en el botón "Upload" para seleccionar una imagen:**

In [15]:
# Widget para subir imagen
upload_widget = widgets.FileUpload(
    accept='image/*',
    multiple=False,
    description='📸 Upload'
)

output_area = widgets.Output()

def on_upload_change(change):
    """Se ejecuta cuando se sube una imagen."""
    with output_area:
        clear_output(wait=True)
        
        # Verificar que hay archivo
        if not upload_widget.value or len(upload_widget.value) == 0:
            return
        
        # Obtener la imagen subida (maneja tanto dict como tuple)
        if isinstance(upload_widget.value, dict):
            # Versión antigua de ipywidgets
            uploaded_file = list(upload_widget.value.values())[0]
            filename = list(upload_widget.value.keys())[0]
        else:
            # Versión nueva de ipywidgets (tupla)
            uploaded_file = upload_widget.value[0]
            filename = uploaded_file['name']
        
        print(f"📂 Archivo: {filename}")
        print("🔄 Procesando...\n")
        
        try:
            # Cargar imagen desde bytes
            img = Image.open(io.BytesIO(uploaded_file['content'])).convert('RGB')
            original_size = img.size
            
            # Preprocesar
            img_resized = img.resize((224, 224))
            img_array = np.array(img_resized) / 255.0
            img_batch = np.expand_dims(img_array, axis=0)
            
            # Predecir
            prediction = model.predict(img_batch, verbose=0)[0][0]
            
            # Interpretar
            is_dusty = prediction > 0.5
            label = '🟠 SUCIO' if is_dusty else '✅ LIMPIO'
            confidence = prediction if is_dusty else 1 - prediction
            
            # Mostrar resultados
            fig, axes = plt.subplots(1, 3, figsize=(15, 5))
            
            # Imagen original
            axes[0].imshow(img)
            axes[0].set_title('Imagen Original', fontsize=14, fontweight='bold')
            axes[0].axis('off')
            
            # Resultado
            axes[1].axis('off')
            color = 'orange' if is_dusty else 'green'
            result_text = f"""
RESULTADO
{'='*30}

Estado: {label}
Confianza: {confidence:.1%}

PROBABILIDADES:
  Limpio: {(1-prediction)*100:.1f}%
  Sucio:  {prediction*100:.1f}%

{'='*30}
            """
            axes[1].text(0.5, 0.5, result_text, 
                         fontsize=11, ha='center', va='center',
                         family='monospace',
                         bbox=dict(boxstyle='round', facecolor=color, alpha=0.2, 
                                  linewidth=2, edgecolor=color))
            
            # Barras de probabilidad
            axes[2].barh(['Limpio', 'Sucio'], [1-prediction, prediction], 
                         color=['green', 'orange'], alpha=0.7, edgecolor='black')
            axes[2].set_xlim(0, 1)
            axes[2].set_xlabel('Probabilidad', fontweight='bold')
            axes[2].set_title('Distribución', fontsize=14, fontweight='bold')
            axes[2].grid(axis='x', alpha=0.3)
            
            # Agregar porcentajes
            for i, (cat, prob) in enumerate(zip(['Limpio', 'Sucio'], [1-prediction, prediction])):
                axes[2].text(prob + 0.02, i, f'{prob*100:.1f}%', 
                            va='center', fontsize=10, fontweight='bold')
            
            plt.suptitle(f'Clasificación: {label}', fontsize=16, fontweight='bold')
            plt.tight_layout()
            plt.show()
            
            # Resumen en texto
            print("\n" + "="*60)
            print("📊 RESUMEN")
            print("="*60)
            print(f"   🎯 Predicción: {label}")
            print(f"   💯 Confianza: {confidence:.1%}")
            print(f"   ✅ Probabilidad Limpio: {(1-prediction):.1%}")
            print(f"   🟠 Probabilidad Sucio: {prediction:.1%}")
            print(f"   📐 Tamaño imagen: {original_size[0]}x{original_size[1]}px")
            print("="*60)
            
            # Recomendación
            if is_dusty:
                if confidence > 0.8:
                    print("\n⚠️  RECOMENDACIÓN: El panel necesita limpieza urgente.")
                else:
                    print("\n💡 RECOMENDACIÓN: Considere revisar y limpiar el panel pronto.")
            else:
                if confidence > 0.8:
                    print("\n✅ RECOMENDACIÓN: El panel está en buenas condiciones.")
                else:
                    print("\n💡 RECOMENDACIÓN: Monitoree el panel regularmente.")
                    
        except Exception as e:
            print(f"❌ Error al procesar la imagen: {e}")
            import traceback
            traceback.print_exc()

# Conectar el evento
upload_widget.observe(on_upload_change, names='value')

# Mostrar widgets
display(upload_widget, output_area)

print("\n✅ Selector listo - Click en 'Upload' para seleccionar una imagen")

FileUpload(value=(), accept='image/*', description='📸 Upload')

Output()


✅ Selector listo - Click en 'Upload' para seleccionar una imagen
